# ClustMRF + Dense Re-ranking on MSMARCO Passages (TREC DL 2019–2023)

Runs the full BM25 → SDM → ClustMRF → E5/BGE pipeline on four TREC Deep Learning **passage** tracks.
Set `DATASET_KEY` in §2 to select the target collection, run all cells, then use §14 to compare across runs.

| `DATASET_KEY` | Corpus | Test set | Queries | Rel. scale |
|---------------|--------|----------|---------|------------|
| `dl19-passage` | MS MARCO V1 Passages (8.8M) | TREC DL 2019 | 43 | 0–3 |
| `dl20-passage` | MS MARCO V1 Passages (8.8M) | TREC DL 2020 | 54 | 0–3 |
| `dl22-passage` | MS MARCO V2 Passages (138M) | TREC DL 2022 | 76 | 0–3 |
| `dl23-passage` | MS MARCO V2 Passages (138M) | TREC DL 2023 | 82 | 0–3 |

**Data:** Downloaded automatically via `ir_datasets` on first run.  
**Index strategy:** Judged passages fetched via `docs_store()` + reservoir-sampled background  
(V1: 50k background; V2: docstore-only by default — set `N_BACKGROUND > 0` to add background).

## 1  Environment Setup

In [21]:
import os, sys, warnings, pathlib, logging, random, time, json, glob, platform
warnings.filterwarnings('ignore')

# ── Java home — Mac vs Linux ───────────────────────────────────────────────────
# Use the unversioned symlink on Linux: the versioned path (e.g. java-11-openjdk-11.0.25.0.9)
# contains "25." which PyTerrier's version-check regex falsely matches as "Java 25",
# causing it to add --enable-native-access=ALL-UNNAMED, which Java 11 rejects.
if platform.system() == 'Darwin':
    os.environ.setdefault(
        'JAVA_HOME',
        '/Library/Java/JavaVirtualMachines/temurin-21.jdk/Contents/Home',
    )
else:
    os.environ.setdefault(
        'JAVA_HOME',
        '/usr/lib/jvm/java-11-openjdk',
    )

# ── ir_datasets cache stays on LOCAL disk ─────────────────────────────────────
# The NFS mount (/mnt/bi-strg4) does not support POSIX file locking (ENOLCK).
# ir_datasets uses filelock (fcntl) for its docstore and downloads, so it must
# live on a local filesystem.  ~/.ir_datasets is the default; leave it alone.
# Terrier indexes and run files go to NFS instead (Java I/O does not use fcntl).

ROOT = pathlib.Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import ir_datasets
import ir_measures
from ir_measures import MAP, nDCG, P
from tqdm.auto import tqdm

import pyterrier as pt
if not pt.java.started():
    pt.java.set_memory_limit(4096)
    pt.java.init()

# ir_datasets uses a custom Logger class whose .logger() lazily initialises with
# setLevel(INFO) on first call, overriding any earlier basicConfig/setLevel call.
# Fix: force-init the logger into ir_datasets' internal cache right now, then
# immediately raise it to WARNING.  Subsequent calls (building docstore, etc.)
# find it in cache, skip the level-reset, and the INFO messages are filtered out.
import ir_datasets.log as _ird_log
_ird_log.Logger(None).logger()             # populate _logger_cache["ir_datasets"]
logging.getLogger('ir_datasets').setLevel(logging.WARNING)
logging.getLogger('terrier').setLevel(logging.WARNING)

MEASURES = [MAP @ 50, P @ 5, nDCG @ 5, nDCG @ 10, nDCG @ 100]

print(f'Platform   : {platform.system()} ({platform.node()})')
print(f'JAVA_HOME  : {os.environ.get("JAVA_HOME", "(not set)")}')
print(f'IR_DS_HOME : {os.environ.get("IR_DATASETS_HOME", "~/.ir_datasets (default, local)")}')
print(f'PyTerrier  : {pt.__version__}')
print(f'ir_datasets: {ir_datasets.__version__}')
print(f'ir_measures: {ir_measures.__version__}')

Platform   : Linux (ieir72.iem.technion.ac.il)
JAVA_HOME  : /usr/lib/jvm/java-11-openjdk
IR_DS_HOME : ~/.ir_datasets (default, local)
PyTerrier  : 1.1.0
ir_datasets: 0.6.1
ir_measures: 0.4.3


## 2  Dataset Selection

Change `DATASET_KEY` to switch between collections. Run §14 after completing all four to compare.

In [22]:
# ── Select dataset ─────────────────────────────────────────────────────────────
# Options: 'dl19-passage', 'dl20-passage', 'dl22-passage', 'dl23-passage'
DATASET_KEY = 'dl19-passage'

# ── Experiment parameters — change here, cache invalidates automatically ───────
N_RESULTS       = 100   # passages retrieved by BM25 / SDM
CLUSTMRF_K      = 5     # ClustMRF neighbourhood size (k nearest neighbours)
CLUSTMRF_N_DOCS = 100   # ClustMRF re-ranking depth
E5_CLUSTER_K    = 5     # E5-Cluster neighbourhood size

# ── Registry ──────────────────────────────────────────────────────────────────
REGISTRY = {
    'dl19-passage': {
        'ir_datasets_id' : 'msmarco-passage/trec-dl-2019/judged',
        'corpus_id'      : 'msmarco-passage',
        'n_background'   : 50_000,
        'description'    : 'TREC DL 2019 Passage (MS MARCO V1, 8.8M passages)',
    },
    'dl20-passage': {
        'ir_datasets_id' : 'msmarco-passage/trec-dl-2020/judged',
        'corpus_id'      : 'msmarco-passage',
        'n_background'   : 50_000,
        'description'    : 'TREC DL 2020 Passage (MS MARCO V1, 8.8M passages)',
    },
    'dl22-passage': {
        'ir_datasets_id' : 'msmarco-passage-v2/trec-dl-2022/judged',
        'corpus_id'      : 'msmarco-passage-v2',
        'n_background'   : 0,
        'description'    : 'TREC DL 2022 Passage (MS MARCO V2, 138M passages)',
    },
    'dl23-passage': {
        'ir_datasets_id' : 'msmarco-passage-v2/trec-dl-2023/judged',
        'corpus_id'      : 'msmarco-passage-v2',
        'n_background'   : 0,
        'description'    : 'TREC DL 2023 Passage (MS MARCO V2, 138M passages)',
    },
}

cfg = REGISTRY[DATASET_KEY]

# ── Data root — prefer large network storage when mounted ─────────────────────
_LARGE_STORAGE = pathlib.Path('/mnt/bi-strg4/pool1-data/u/avi.simkin')
DATA_ROOT = _LARGE_STORAGE if _LARGE_STORAGE.is_dir() else ROOT / 'data'

INDEX_DIR = DATA_ROOT / 'indexes' / DATASET_KEY
RUNS_DIR  = DATA_ROOT / 'runs'    / DATASET_KEY
INDEX_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True,  exist_ok=True)

print(f'Dataset    : {cfg["description"]}')
print(f'ir_ds ID   : {cfg["ir_datasets_id"]}')
print(f'N_RESULTS  : {N_RESULTS}')
print(f'Data root  : {DATA_ROOT}')
print(f'Index      : {INDEX_DIR}')
print(f'Runs       : {RUNS_DIR}')

Dataset    : TREC DL 2019 Passage (MS MARCO V1, 8.8M passages)
ir_ds ID   : msmarco-passage/trec-dl-2019/judged
N_RESULTS  : 100
Data root  : /mnt/bi-strg4/pool1-data/u/avi.simkin
Index      : /mnt/bi-strg4/pool1-data/u/avi.simkin/indexes/dl19-passage
Runs       : /mnt/bi-strg4/pool1-data/u/avi.simkin/runs/dl19-passage


## 3  Load Queries and Qrels

`ir_datasets` downloads data on first access and caches it under `~/.ir_datasets/`.

In [23]:
print(f'Loading {cfg["ir_datasets_id"]} …')
ds = ir_datasets.load(cfg['ir_datasets_id'])

queries_df = pd.DataFrame([
    {'qid': q.query_id, 'query': q.text}
    for q in ds.queries_iter()
]).sort_values('qid').reset_index(drop=True)

qrels_rows  = []
qrel_docids = set()
for qrel in ds.qrels_iter():
    qrels_rows.append({
        'query_id'  : qrel.query_id,
        'doc_id'    : qrel.doc_id,
        'relevance' : int(qrel.relevance),
    })
    qrel_docids.add(qrel.doc_id)

qrels_df = pd.DataFrame(qrels_rows)

print(f'Queries           : {len(queries_df)}')
print(f'Qrel rows         : {len(qrels_df):,}')
print(f'Unique judged docs : {len(qrel_docids):,}')
print(f'\nRelevance distribution:')
print(qrels_df['relevance'].value_counts().sort_index().to_string())
queries_df.head()

Loading msmarco-passage/trec-dl-2019/judged …
Queries           : 43
Qrel rows         : 9,260
Unique judged docs : 9,139

Relevance distribution:
relevance
0    5158
1    1601
2    1804
3     697


,qid,query
0,1037798,who is robert gray
1,104861,cost of interior concrete flooring
2,1063750,why did the us volunterilay enter ww1
3,1103812,who formed the commonwealth of independent states
4,1106007,define visceral?


## 4  MSMARCO Data Usage Agreement\n\nir_datasets will prompt you to **agree to the MSMARCO terms** the first time the corpus is accessed.\nType `agree` when prompted and press Enter. The agreement is cached under `~/.ir_datasets/` so this only happens once per corpus version."
</invoke>

In [24]:
# Trigger the data usage agreement and fully warm up the docstore before indexing.
# If running for the first time: type 'agree' when prompted, then press Enter.
print(f'Opening corpus: {cfg["corpus_id"]}')
corpus_ds = ir_datasets.load(cfg['corpus_id'])
docstore  = corpus_ds.docs_store()   # <-- agreement prompt fires here on first run

# Warm-up: docstore may load its index lazily on the first .get() call,
# which would fire a tqdm bar inside our loop.  Trigger it here instead.
_first = next(iter(sorted(qrel_docids)))
_ = docstore.get(_first)
del _first, _
print('Corpus accessible — agreement confirmed, docstore warm.')

Opening corpus: msmarco-passage
Corpus accessible — agreement confirmed, docstore warm.


## 4  Build Judgment-Pool Index

**Strategy:**
1. Fetch all judged passages via `docs_store()` (fast O(1) lookup — no corpus streaming needed).
2. Optionally reservoir-sample background passages for realistic BM25 competition  
   (V1: 50k background by streaming the 8.8M corpus; V2: disabled by default).

**First run** downloads and caches the corpus via ir_datasets, then builds the Terrier index.  
**Subsequent runs** load the cached index in seconds.

In [25]:
import shutil

props_file    = INDEX_DIR / 'data.properties'
blocks_marker = INDEX_DIR / '.blocks_enabled'

if props_file.exists() and not blocks_marker.exists():
    print('Existing index lacks block info (needed for SDM). Removing and rebuilding…')
    shutil.rmtree(str(INDEX_DIR))
    INDEX_DIR.mkdir(parents=True, exist_ok=True)

if not props_file.exists():
    # ── Step 1: fetch judged passages by ID ───────────────────────────────────
    # docstore was fully warmed up in §4 so no lazy-load tqdm fires here.
    print(f'Fetching {len(qrel_docids):,} judged passages via docstore…')
    pool_docs, missing = [], 0
    for i, doc_id in enumerate(sorted(qrel_docids)):
        doc = docstore.get(doc_id)
        if doc is not None:
            pool_docs.append({'docno': doc.doc_id, 'text': doc.text or ''})
        else:
            missing += 1
        if (i + 1) % 10_000 == 0:
            print(f'  {i + 1:,} / {len(qrel_docids):,}')
    print(f'  done — found {len(pool_docs):,}  missing {missing}')

    # ── Step 2: optional background sample ────────────────────────────────────
    # Do NOT wrap docs_iter() in a second tqdm — ir_datasets already provides
    # its own progress bar, and nesting tqdm bars leaves blank lines in Jupyter.
    n_bg = cfg['n_background']
    if n_bg > 0:
        print(f'Sampling {n_bg:,} background passages (streaming corpus)…')
        rng = random.Random(42)
        reservoir, n_seen = [], 0
        for doc in corpus_ds.docs_iter():        # ir_datasets' own bar is enough
            if doc.doc_id not in qrel_docids:
                n_seen += 1
                rec = {'docno': doc.doc_id, 'text': doc.text or ''}
                if len(reservoir) < n_bg:
                    reservoir.append(rec)
                else:
                    j = rng.randrange(n_seen)
                    if j < n_bg:
                        reservoir[j] = rec
        print(f'  sampled {len(reservoir):,} background passages')
        pool_docs.extend(reservoir)
    else:
        print('Background sampling disabled (n_background=0).')

    print(f'Total pool: {len(pool_docs):,} passages  →  building index…')

    # ── Step 3: index ─────────────────────────────────────────────────────────
    indexer = pt.IterDictIndexer(
        str(INDEX_DIR),
        overwrite  = True,
        meta       = {'docno': 40, 'text': 4096},
        text_attrs = ['text'],
        blocks     = True,
        tokeniser  = 'EnglishTokeniser',
        stemmer    = 'PorterStemmer',
        stopwords  = 'terrier',
    )
    indexer.index(iter(pool_docs))
    blocks_marker.touch()
    print('Index built.')
else:
    print('Cached index found — loading.')

index = pt.IndexFactory.of(str(props_file))
stats = index.getCollectionStatistics()
print(f'Documents    : {stats.numberOfDocuments:,}')
print(f'Tokens       : {stats.numberOfTokens:,}')
print(f'Unique terms : {stats.numberOfUniqueTerms:,}')

Cached index found — loading.
Documents    : 59,139
Tokens       : 1,927,404
Unique terms : 67,235


## 5  Initial Retrieval: BM25 and SDM

Both retrievers return top-100 passages with `text` metadata stored.  
SDM rewrite produces a `query_0` column (original NL query) used by dense re-rankers in §8.

In [26]:
bm25_br = pt.BatchRetrieve(
    index,
    wmodel      = 'BM25',
    num_results = N_RESULTS,
    metadata    = ['docno', 'text'],
    controls    = {'bm25.b': 0.75, 'bm25.k_1': 1.2},
    verbose     = True,
)

print('Running BM25…')
bm25_run = bm25_br.transform(queries_df)
print(f'BM25: {len(bm25_run):,} rows  ({bm25_run.qid.nunique()} queries)')

sdm_rewrite  = pt.rewrite.SDM()
dirichlet_br = pt.BatchRetrieve(
    index,
    wmodel      = 'DirichletLM',
    num_results = N_RESULTS,
    metadata    = ['docno', 'text'],
    controls    = {'c': 2500},
    verbose     = True,
)
sdm_pipe = sdm_rewrite >> dirichlet_br

print('\nRunning SDM + DirichletLM…')
sdm_run = sdm_pipe.transform(queries_df)
print(f'SDM: {len(sdm_run):,} rows  ({sdm_run.qid.nunique()} queries)')

assert 'query_0' in sdm_run.columns, 'Expected query_0 column from SDM rewrite'
sdm_run.head()

Running BM25…


TerrierRetr(BM25):   0%|          | 0/43 [00:00<?, ?q/s]

BM25: 4,205 rows  (43 queries)

Running SDM + DirichletLM…


TerrierRetr(DirichletLM):   0%|          | 0/43 [00:00<?, ?q/s]

SDM: 4,205 rows  (43 queries)


,qid,query,query_0,docid,docno,text,rank,score
0,1037798,who robert gray #combine:0=0.1:wmodel=org.terr...,who is robert gray,3131,3641634,"Captain Robert Gray, May 1972. Discovering the...",0,6.761250
1,1037798,who robert gray #combine:0=0.1:wmodel=org.terr...,who is robert gray,8862,8760873,Team Mississippi Robert Gray For Governor Offi...,1,6.180325
2,1037798,who robert gray #combine:0=0.1:wmodel=org.terr...,who is robert gray,8856,8760864,Team Mississippi Robert Gray For Governor Offi...,2,6.038735
3,1037798,who robert gray #combine:0=0.1:wmodel=org.terr...,who is robert gray,3112,3620983,"I'm not a politician, said Gray in a Wednesday...",3,5.819977
4,1037798,who robert gray #combine:0=0.1:wmodel=org.terr...,who is robert gray,8858,8760867,Robert Gray. A surprise came on the Democratic...,4,5.479296


## 6  Baseline Evaluation (BM25 & SDM)

In [27]:
bm25_eval = bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
bm25_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, bm25_eval)

sdm_eval = sdm_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
sdm_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, sdm_eval)

print(f'Baselines — {cfg["description"]}')
print(f'{"Measure":<20} {"BM25":>10} {"SDM":>10}')
print('=' * 42)
for m in MEASURES:
    print(f'  {str(m):<18} {float(bm25_agg[m]):>10.4f} {float(sdm_agg[m]):>10.4f}')

Baselines — TREC DL 2019 Passage (MS MARCO V1, 8.8M passages)
Measure                    BM25        SDM
  AP@50                  0.2798     0.2690
  P@5                    0.6140     0.5953
  nDCG@5                 0.4763     0.4450
  nDCG@10                0.4772     0.4645
  nDCG@100               0.6012     0.5875


In [28]:
def save_trec_run(run_df: pd.DataFrame, path: pathlib.Path, tag: str) -> None:
    with open(path, 'w') as f:
        for qid, group in run_df.sort_values(['qid', 'rank']).groupby('qid'):
            for rank, row in enumerate(group.itertuples(), start=1):
                f.write(f'{qid} Q0 {row.docno} {rank} {row.score:.6f} {tag}\n')


def load_trec_run(path: pathlib.Path) -> pd.DataFrame:
    rows = []
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) >= 5:
                rows.append({'qid': p[0], 'docno': p[2],
                             'rank': int(p[3]), 'score': float(p[4])})
    return pd.DataFrame(rows)


def run_cached(path: pathlib.Path, config: dict,
               compute_fn, tag: str) -> pd.DataFrame:
    """Load run from cache when config matches; otherwise compute, save, and
    store the config so future calls can detect when parameters change."""
    cfg_path = path.with_suffix('.config.json')
    if path.exists() and cfg_path.exists():
        with open(cfg_path) as f:
            stored = json.load(f)
        if stored == config:
            print(f'Cache hit  → {path.name}')
            return load_trec_run(path)
        changed = sorted(
            {k for k in config if config.get(k) != stored.get(k)} |
            {k for k in stored  if k not in config}
        )
        print(f'Config changed ({", ".join(changed)}) — re-running {path.name}…')
    run_df = compute_fn()
    save_trec_run(run_df, path, tag)
    with open(cfg_path, 'w') as f:
        json.dump(config, f, indent=2)
    return run_df

## 7  ClustMRF Re-ranking

Re-ranks the SDM top-100 using the MRF cluster-scoring framework (Raiber & Kurland, SIGIR 2013).  
Default weights from Table 3 (non-web setting); k=5 cluster size.

In [29]:
from src.algorithms.clust_mrf import ClustMRF

clustmrf = ClustMRF(index=index, k=CLUSTMRF_K, n_docs=CLUSTMRF_N_DOCS)

_cfg = {'n_results': N_RESULTS, 'k': CLUSTMRF_K, 'n_docs': CLUSTMRF_N_DOCS}
t0   = time.time()
clustmrf_run = run_cached(
    RUNS_DIR / 'clustmrf.txt', _cfg,
    lambda: clustmrf.transform(sdm_run), 'ClustMRF',
)
if time.time() - t0 > 1:
    print(f'Done in {time.time()-t0:.1f}s')

clustmrf_eval = clustmrf_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
clustmrf_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, clustmrf_eval)

rows = []
for name, agg in [('BM25', bm25_agg), ('SDM', sdm_agg), ('ClustMRF', clustmrf_agg)]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    rows.append(row)

res_table = pd.DataFrame(rows).set_index('System')
res_table.loc['Δ ClustMRF−SDM'] = res_table.loc['ClustMRF'] - res_table.loc['SDM']
print(f'\n=== {DATASET_KEY} ===\n{res_table.to_string()}')

Cache hit  → clustmrf.txt

=== dl19-passage ===
                 AP@50     P@5  nDCG@5  nDCG@10  nDCG@100
System                                                   
BM25            0.2798  0.6140  0.4763   0.4772    0.6012
SDM             0.2690  0.5953  0.4450   0.4645    0.5875
ClustMRF        0.2534  0.5721  0.4282   0.4479    0.5798
Δ ClustMRF−SDM -0.0156 -0.0232 -0.0168  -0.0166   -0.0077


## 8  Dense Re-ranking: E5 and BGE

Both models re-rank the SDM top-100 using cosine similarity between mean-pooled query and passage vectors.

| Model | Query prefix | Passage prefix |
|-------|-------------|----------------|
| `intfloat/e5-base-v2` | `query: ` | `passage: ` |
| `BAAI/bge-base-en-v1.5` | `Represent this sentence: ` | *(none)* |

In [30]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODE_BATCH = 64


def mean_pool_encode(model, tokenizer, texts: list) -> np.ndarray:
    enc = tokenizer(texts, padding=True, truncation=True, max_length=512,
                    return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        out = model(**enc)
    mask   = enc['attention_mask'].unsqueeze(-1).float()
    pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return F.normalize(pooled, dim=-1).cpu().float().numpy()


def dense_rerank(run_df, model, tokenizer,
                 query_col='query_0', query_prefix='', doc_prefix=''):
    results = []
    for qid, group in tqdm(run_df.groupby('qid'), desc='queries', leave=True):
        query  = query_prefix + str(group[query_col].iloc[0])
        docs   = [doc_prefix + t for t in group['text'].fillna('').tolist()]
        d_vecs = np.vstack([
            mean_pool_encode(model, tokenizer, docs[i:i + ENCODE_BATCH])
            for i in range(0, len(docs), ENCODE_BATCH)
        ])
        q_vec  = mean_pool_encode(model, tokenizer, [query])
        scores = (d_vecs @ q_vec.T).squeeze(-1)
        grp_out = group.copy()
        grp_out['score'] = scores
        grp_out = grp_out.sort_values('score', ascending=False)
        grp_out['rank'] = range(1, len(grp_out) + 1)
        results.append(grp_out)
    return pd.concat(results, ignore_index=True)


print(f'Device: {DEVICE}')

print('\nLoading intfloat/e5-base-v2…')
e5_tok   = AutoTokenizer.from_pretrained('intfloat/e5-base-v2')
e5_model = AutoModel.from_pretrained('intfloat/e5-base-v2').to(DEVICE).eval()

print('Loading BAAI/bge-base-en-v1.5…')
bge_tok   = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')
bge_model = AutoModel.from_pretrained('BAAI/bge-base-en-v1.5').to(DEVICE).eval()

e5_run = run_cached(
    RUNS_DIR / 'e5.txt',
    {'n_results': N_RESULTS, 'model': 'intfloat/e5-base-v2'},
    lambda: dense_rerank(sdm_run, e5_model, e5_tok,
                         query_prefix='query: ', doc_prefix='passage: '),
    'E5_base_v2',
)

bge_run = run_cached(
    RUNS_DIR / 'bge.txt',
    {'n_results': N_RESULTS, 'model': 'BAAI/bge-base-en-v1.5'},
    lambda: dense_rerank(sdm_run, bge_model, bge_tok,
                         query_prefix='Represent this sentence: '),
    'BGE_base_v15',
)

e5_eval  = e5_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
bge_eval = bge_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
e5_agg   = ir_measures.calc_aggregate(MEASURES, qrels_df, e5_eval)
bge_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, bge_eval)

dense_rows = []
for name, agg in [
    ('BM25',             bm25_agg),
    ('SDM',              sdm_agg),
    ('ClustMRF',         clustmrf_agg),
    ('E5-base-v2',       e5_agg),
    ('BGE-base-en-v1.5', bge_agg),
]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    dense_rows.append(row)

dense_df = pd.DataFrame(dense_rows).set_index('System')
print(f'\n=== {DATASET_KEY}: Dense Re-ranking ===')
print(dense_df.to_string())

Device: cpu

Loading intfloat/e5-base-v2…
Loading BAAI/bge-base-en-v1.5…
Cache hit  → e5.txt
Cache hit  → bge.txt

=== dl19-passage: Dense Re-ranking ===
                   AP@50     P@5  nDCG@5  nDCG@10  nDCG@100
System                                                     
BM25              0.2798  0.6140  0.4763   0.4772    0.6012
SDM               0.2690  0.5953  0.4450   0.4645    0.5875
ClustMRF          0.2534  0.5721  0.4282   0.4479    0.5798
E5-base-v2        0.4060  0.8465  0.7224   0.7131    0.6849
BGE-base-en-v1.5  0.3937  0.8372  0.7293   0.7226    0.6814


### 8.1  E5 Cluster Re-ranking

Clusters the SDM top-100 passages into **k=5** groups using K-means on E5 embeddings.
Clusters are ordered by centroid–query cosine similarity (closest first); passages within
each cluster are ordered by individual passage–query cosine similarity.

In [31]:
import importlib
import src.algorithms.e5_cluster_reranker as _e5cr_mod
importlib.reload(_e5cr_mod)
from src.algorithms.e5_cluster_reranker import E5ClusterReranker

e5_cluster = E5ClusterReranker(e5_model, e5_tok, k=E5_CLUSTER_K, mode='knn', device=DEVICE)

e5_cluster_run = run_cached(
    RUNS_DIR / 'e5_cluster.txt',
    {'n_results': N_RESULTS, 'model': 'intfloat/e5-base-v2',
     'k': E5_CLUSTER_K, 'mode': 'knn'},
    lambda: e5_cluster.transform(sdm_run),
    'E5_Cluster_knn',
)

e5_cluster_eval = e5_cluster_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
e5_cluster_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, e5_cluster_eval)

all_rows = []
for name, agg in [
    ('BM25',              bm25_agg),
    ('SDM',               sdm_agg),
    ('ClustMRF',          clustmrf_agg),
    ('E5-base-v2',        e5_agg),
    ('BGE-base-en-v1.5',  bge_agg),
    ('E5-Cluster knn=5',  e5_cluster_agg),
]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    all_rows.append(row)

all_df = pd.DataFrame(all_rows).set_index('System')
print(f'\n=== {DATASET_KEY}: All Re-rankers ===')
print(all_df.to_string())

Cache hit  → e5_cluster.txt

=== dl19-passage: All Re-rankers ===
                   AP@50     P@5  nDCG@5  nDCG@10  nDCG@100
System                                                     
BM25              0.2798  0.6140  0.4763   0.4772    0.6012
SDM               0.2690  0.5953  0.4450   0.4645    0.5875
ClustMRF          0.2534  0.5721  0.4282   0.4479    0.5798
E5-base-v2        0.4060  0.8465  0.7224   0.7131    0.6849
BGE-base-en-v1.5  0.3937  0.8372  0.7293   0.7226    0.6814
E5-Cluster knn=5  0.3478  0.7674  0.6419   0.6439    0.6606


### 8.2  K-Means Cluster Re-ranking (E5 & BGE × centroid types)

Runs six configurations: **E5** and **BGE** embeddings × three centroid types
(`mean`, `medoid`, `query_weighted`), all with K-means k=5.

| Centroid type | Description |
|---|---|
| `mean` | Arithmetic mean of member embeddings, L2-normalised |
| `medoid` | Member whose embedding is closest to the arithmetic mean |
| `query_weighted` | Mean weighted by softmax(passage–query cosine similarities) |

In [32]:
import importlib
import src.algorithms.e5_cluster_reranker as _e5cr_mod
importlib.reload(_e5cr_mod)
from src.algorithms.e5_cluster_reranker import E5ClusterReranker

KMEANS_CONFIGS = [
    # (system_name, model, tokenizer, query_prefix, doc_prefix, centroid_type)
    ('E5-kmeans mean',       e5_model,  e5_tok,  'query: ',                   'passage: ', 'mean'),
    ('E5-kmeans medoid',     e5_model,  e5_tok,  'query: ',                   'passage: ', 'medoid'),
    ('E5-kmeans q-weighted', e5_model,  e5_tok,  'query: ',                   'passage: ', 'query_weighted'),
    ('BGE-kmeans mean',      bge_model, bge_tok, 'Represent this sentence: ', '',          'mean'),
    ('BGE-kmeans medoid',    bge_model, bge_tok, 'Represent this sentence: ', '',          'medoid'),
    ('BGE-kmeans q-weighted',bge_model, bge_tok, 'Represent this sentence: ', '',          'query_weighted'),
]

kmeans_runs = {}
kmeans_aggs = {}

for name, mdl, tok, q_pfx, d_pfx, c_type in KMEANS_CONFIGS:
    file_tag  = name.replace(' ', '_').replace('-', '_')
    cache_cfg = {
        'n_results'    : N_RESULTS,
        'model'        : 'intfloat/e5-base-v2' if mdl is e5_model else 'BAAI/bge-base-en-v1.5',
        'k'            : E5_CLUSTER_K,
        'mode'         : 'kmeans',
        'centroid_type': c_type,
    }
    reranker = E5ClusterReranker(
        mdl, tok, k=E5_CLUSTER_K, mode='kmeans',
        centroid_type=c_type, query_prefix=q_pfx, doc_prefix=d_pfx,
        device=DEVICE,
    )
    run = run_cached(
        RUNS_DIR / f'{file_tag}.txt',
        cache_cfg,
        lambda r=reranker: r.transform(sdm_run),
        file_tag,
    )
    kmeans_runs[name] = run
    ev = run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
    kmeans_aggs[name] = ir_measures.calc_aggregate(MEASURES, qrels_df, ev)

# ── Summary table ──────────────────────────────────────────────────────────────
all_rows = []
for sname, agg in [
    ('BM25',             bm25_agg),
    ('SDM',              sdm_agg),
    ('E5-Cluster knn=5', e5_cluster_agg),
    *[(n, kmeans_aggs[n]) for n, *_ in KMEANS_CONFIGS],
]:
    row = {'System': sname}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    all_rows.append(row)

print(f'\n=== {DATASET_KEY}: K-Means Cluster Re-ranking ===')
print(pd.DataFrame(all_rows).set_index('System').to_string())

ClusterRerank (kmeans, mean):   0%|          | 0/43 [00:00<?, ?it/s]

ClusterRerank (kmeans, medoid):   0%|          | 0/43 [00:00<?, ?it/s]

ClusterRerank (kmeans, query_weighted):   0%|          | 0/43 [00:00<?, ?it/s]

ClusterRerank (kmeans, mean):   0%|          | 0/43 [00:00<?, ?it/s]

ClusterRerank (kmeans, medoid):   0%|          | 0/43 [00:00<?, ?it/s]

ClusterRerank (kmeans, query_weighted):   0%|          | 0/43 [00:00<?, ?it/s]


=== dl19-passage: K-Means Cluster Re-ranking ===
                        AP@50     P@5  nDCG@5  nDCG@10  nDCG@100
System                                                          
BM25                   0.2798  0.6140  0.4763   0.4772    0.6012
SDM                    0.2690  0.5953  0.4450   0.4645    0.5875
E5-Cluster knn=5       0.3478  0.7674  0.6419   0.6439    0.6606
E5-kmeans mean         0.3733  0.8047  0.6664   0.6405    0.6591
E5-kmeans medoid       0.3747  0.8093  0.6544   0.6321    0.6550
E5-kmeans q-weighted   0.3733  0.8047  0.6664   0.6405    0.6591
BGE-kmeans mean        0.3446  0.7814  0.6498   0.6337    0.6482
BGE-kmeans medoid      0.3499  0.7535  0.6449   0.6288    0.6493
BGE-kmeans q-weighted  0.3446  0.7814  0.6498   0.6337    0.6481


## 9  Interpolation: ClustMRF + BM25 / SDM

```
score(d, q) = α · score_ClustMRF + (1 − α) · score_base
```
Scores are min-max normalised per query before blending.  Only docs in both runs are retained.

In [33]:
def minmax_norm(run_df: pd.DataFrame) -> pd.DataFrame:
    run_df = run_df.copy()
    grp = run_df.groupby('qid')['score']
    lo  = grp.transform('min')
    hi  = grp.transform('max')
    run_df['score'] = (run_df['score'] - lo) / (hi - lo).replace(0, 1.0)
    return run_df


def interpolate_runs(run_a: pd.DataFrame, run_b: pd.DataFrame,
                     alpha: float) -> pd.DataFrame:
    a = run_a[['qid', 'docno', 'score']].rename(columns={'score': 'score_a'})
    b = run_b[['qid', 'docno', 'score']].rename(columns={'score': 'score_b'})
    merged = pd.merge(a, b, on=['qid', 'docno'])
    merged['score'] = alpha * merged['score_a'] + (1.0 - alpha) * merged['score_b']
    merged = (merged[['qid', 'docno', 'score']]
              .sort_values(['qid', 'score'], ascending=[True, False]))
    merged['rank'] = merged.groupby('qid').cumcount() + 1
    return merged.reset_index(drop=True)


bm25_norm     = minmax_norm(bm25_run)
sdm_norm      = minmax_norm(sdm_run)
clustmrf_norm = minmax_norm(clustmrf_run)

ALPHAS = [0.1, 0.25, 0.5, 0.75, 0.9]

interp_rows = []
for name, agg in [('BM25', bm25_agg), ('SDM', sdm_agg), ('ClustMRF', clustmrf_agg)]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    interp_rows.append(row)
interp_rows.append({'System': '---', **{str(m): '---' for m in MEASURES}})

for alpha in ALPHAS:
    for tag, run_b in [('BM25', bm25_norm), ('SDM', sdm_norm)]:
        run_i = interpolate_runs(clustmrf_norm, run_b, alpha)
        ev_i  = run_i.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
        agg_i = ir_measures.calc_aggregate(MEASURES, qrels_df, ev_i)
        row   = {'System': f'ClustMRF+{tag}  \u03b1={alpha}'}
        for m in MEASURES:
            row[str(m)] = round(float(agg_i[m]), 4)
        interp_rows.append(row)

interp_df = pd.DataFrame(interp_rows).set_index('System')
print(f'=== {DATASET_KEY}: Interpolation (\u03b1 = weight on ClustMRF) ===')
print(interp_df.to_string())

=== dl19-passage: Interpolation (α = weight on ClustMRF) ===
                        AP@50     P@5  nDCG@5 nDCG@10 nDCG@100
System                                                        
BM25                   0.2798   0.614  0.4763  0.4772   0.6012
SDM                     0.269  0.5953   0.445  0.4645   0.5875
ClustMRF               0.2534  0.5721  0.4282  0.4479   0.5798
---                       ---     ---     ---     ---      ---
ClustMRF+BM25  α=0.1    0.286  0.6279  0.4931  0.4809    0.568
ClustMRF+SDM  α=0.1    0.2714   0.614  0.4489  0.4589   0.5877
ClustMRF+BM25  α=0.25  0.2903  0.6372  0.4997  0.4962   0.5729
ClustMRF+SDM  α=0.25   0.2732  0.6093  0.4486  0.4754   0.5898
ClustMRF+BM25  α=0.5   0.2882  0.6558  0.4841  0.4944   0.5687
ClustMRF+SDM  α=0.5    0.2732  0.6093  0.4563  0.4765   0.5922
ClustMRF+BM25  α=0.75  0.2778  0.6233  0.4661  0.4795   0.5617
ClustMRF+SDM  α=0.75   0.2646  0.6233  0.4595  0.4684   0.5877
ClustMRF+BM25  α=0.9   0.2706  0.6093  0.4507  0.4622   0

## 10  Save Results

In [34]:
# TREC run files — saved inline when each run cell executes;
# re-save here in case §7/§8/§8.1/§8.2 were skipped after loading from cache.
save_trec_run(bm25_run,         RUNS_DIR / 'bm25.txt',         'BM25')
save_trec_run(sdm_run,          RUNS_DIR / 'sdm.txt',          'SDM')
save_trec_run(clustmrf_run,     RUNS_DIR / 'clustmrf.txt',     'ClustMRF')
save_trec_run(e5_run,           RUNS_DIR / 'e5.txt',           'E5_base_v2')
save_trec_run(bge_run,          RUNS_DIR / 'bge.txt',          'BGE_base_v15')
save_trec_run(e5_cluster_run,   RUNS_DIR / 'e5_cluster.txt',   'E5_Cluster_knn5')
for name, run in kmeans_runs.items():
    tag = name.replace(' ', '_').replace('-', '_')
    save_trec_run(run, RUNS_DIR / f'{tag}.txt', tag)

# Aggregate results JSON (used by §11 cross-dataset comparison)
results_path = RUNS_DIR / 'results.json'
payload = {
    'dataset'     : DATASET_KEY,
    'description' : cfg['description'],
    'measures'    : [str(m) for m in MEASURES],
    'results'     : {
        name: {str(m): round(float(agg[m]), 4) for m in MEASURES}
        for name, agg in [
            ('BM25',             bm25_agg),
            ('SDM',              sdm_agg),
            ('ClustMRF',         clustmrf_agg),
            ('E5-base-v2',       e5_agg),
            ('BGE-base-en-v1.5', bge_agg),
            ('E5-Cluster knn=5', e5_cluster_agg),
            *[(n, kmeans_aggs[n]) for n, *_ in KMEANS_CONFIGS],
        ]
    }
}
with open(results_path, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Aggregates saved → {results_path.relative_to(DATA_ROOT)}')

Aggregates saved → runs/dl19-passage/results.json


## 11  Cross-Dataset Comparison

Run §2–10 for each `DATASET_KEY`, then execute this cell to compare all saved results side-by-side.

In [35]:
saved = {}
for path in sorted((DATA_ROOT / 'runs').glob('*/results.json')):
    with open(path) as f:
        d = json.load(f)
    saved[d['dataset']] = d

if not saved:
    print('No saved results yet. Complete §10 for at least one DATASET_KEY first.')
else:
    print(f'Loaded results for: {list(saved.keys())}\n')
    SYSTEMS = [
        'BM25', 'SDM', 'ClustMRF',
        'E5-base-v2', 'BGE-base-en-v1.5',
        'E5-Cluster knn=5',
        'E5-kmeans mean', 'E5-kmeans medoid', 'E5-kmeans q-weighted',
        'BGE-kmeans mean', 'BGE-kmeans medoid', 'BGE-kmeans q-weighted',
    ]

    for metric in ['AP@50', 'nDCG@10', 'nDCG@5', 'P@5']:
        rows = []
        for ds_key, d in saved.items():
            row = {'Dataset': ds_key}
            for sys in SYSTEMS:
                row[sys] = d['results'].get(sys, {}).get(metric, float('nan'))
            rows.append(row)
        df = pd.DataFrame(rows).set_index('Dataset')
        print(f'--- {metric} ---')
        print(df.to_string())
        print()

Loaded results for: ['dl19-passage']

--- AP@50 ---
                BM25    SDM  ClustMRF  E5-base-v2  BGE-base-en-v1.5  E5-Cluster knn=5  E5-kmeans mean  E5-kmeans medoid  E5-kmeans q-weighted  BGE-kmeans mean  BGE-kmeans medoid  BGE-kmeans q-weighted
Dataset                                                                                                                                                                                                 
dl19-passage  0.2798  0.269    0.2534       0.406            0.3937            0.3478          0.3733            0.3747                0.3733           0.3446             0.3499                 0.3446

--- nDCG@10 ---
                BM25     SDM  ClustMRF  E5-base-v2  BGE-base-en-v1.5  E5-Cluster knn=5  E5-kmeans mean  E5-kmeans medoid  E5-kmeans q-weighted  BGE-kmeans mean  BGE-kmeans medoid  BGE-kmeans q-weighted
Dataset                                                                                                                       